# Epic Clarity — Note Hydration

Populates `_exponent.omop_epic.note` from Epic Clarity clinical notes.

## Source Tables
- `_exponent._bronze_epic_clarity.hno_info` — note metadata: `NOTE_ID`, `PAT_ID`, `PAT_ENC_CSN_ID`, `ENTRY_DATETIME`, `NOTE_DESC`, `IP_NOTE_TYPE_C`, `NOTE_TYPE_NOADD_C`, `DELETED_CAT_C`, `DELETE_FLAG`
- `_exponent._bronze_epic_clarity.zc_note_type_ip` — inpatient note type reference: `TYPE_IP_C` → `NAME`

## Patient Join
- `hno_info.PAT_ID` joins to `omop_mapping.source_to_person` via Epic person key:
  `CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', PAT_ID)`

## Pipeline
1. `silver_note` — staged temp view, full OMOP field set
2. MERGE → `omop_silver.note`
3. INSERT → `omop_mapping.source_to_note`
4. `gold` — resolves surrogate IDs and FK references
5. MERGE → `omop_epic.note`

## Filters
- `DELETED_CAT_C IS NULL OR DELETED_CAT_C <> 2` — excludes Epic-deleted notes (DELETED_CAT_C=2 = 3,761 rows)
- `DELETE_FLAG = 0` — excludes ETL-flagged deletes (DELETE_FLAG=1 = 15,263 rows)
- `PAT_ID IS NOT NULL` — excludes orphaned note records
- `ENTRY_DATETIME IS NOT NULL` — excludes undated notes

## Known Gaps
- `note_text` is stubbed with `NOTE_DESC` (note title/description field) — `hno_note_text` is not present in the bronze extract. Full note body text requires re-extraction from Epic Clarity (client follow-up needed).
- `notes_link_ord_txn` links notes to orders via `LINKED_ORD_ID` — not used here; no order-to-visit mapping available yet.

## Dependencies
- `omop_mapping.source_to_person` must be populated for `epic_clarity`
- `omop_mapping.source_to_visit_occurrence` must be populated for `epic_clarity`

In [ ]:
%sql
-- TRUNCATE Gold table for Epic (run this to clear stale data before reload)
-- TRUNCATE TABLE _exponent.omop_epic.note;

In [ ]:
%sql
-- Delete Epic records from Silver and Mapping tables (for full refresh)
DELETE FROM _exponent.omop_silver.note WHERE source_system = 'epic_clarity';

DELETE FROM _exponent.omop_mapping.source_to_note WHERE source_system = 'epic_clarity';

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_note AS
SELECT
  CONCAT_WS(
    CHR(31),
    'epic_clarity',
    'hno_info',
    'NOTE_ID',
    h.NOTE_ID
  )                                                         AS note_source_value,

  stp.person_id                                             AS person_id,

  CAST(h.ENTRY_DATETIME AS DATE)                            AS note_date,
  h.ENTRY_DATETIME                                          AS note_datetime,

  -- Resolve note_type_concept_id from mapping; default to 44814645 (EHR Note) if no mapping found
  COALESCE(
    nt_map.omop_concept_id,
    44814645  --- Default to EHR Note if no mapping found
  )                                                         AS note_type_concept_id,

  -- Resolve note_class_concept_id: always Clinical Note (concept 44814645)
  44814645                                                  AS note_class_concept_id,

  -- note_title: use NOTE_DESC (note description / title field from hno_info)
  h.NOTE_DESC                                               AS note_title,

  -- note_text: stubbed with NOTE_DESC — hno_note_text not present in bronze extract
  -- Full note body text requires re-extraction from Epic Clarity (client follow-up needed)
  COALESCE(REGEXP_REPLACE(h.NOTE_DESC, '[\\x00-\\x1F\\x7F]', ' '), '')  AS note_text,

  32678                                                     AS encoding_concept_id,    --- UTF-8
  4180186                                                   AS language_concept_id,    --- English

  NULL                                                      AS provider_id,

  -- Visit staging key: PAT_ENC_CSN_ID links to Epic visit_occurrence
  CASE
    WHEN h.PAT_ENC_CSN_ID IS NOT NULL AND h.PAT_ENC_CSN_ID <> 0
    THEN CONCAT_WS(
           CHR(31),
           'epic_clarity',
           'PAT_ENC',
           'PAT_ENC_CSN_ID',
           CAST(CAST(h.PAT_ENC_CSN_ID AS BIGINT) AS STRING)
         )
    ELSE NULL
  END                                                       AS visit_occurrence_source_value,

  NULL                                                      AS visit_detail_id,

  NULL                                                      AS note_event_id,
  NULL                                                      AS note_event_field_concept_id,

  'epic_clarity'                                            AS source_system

FROM _exponent._bronze_epic_clarity.hno_info h

-- Join note type reference for inpatient notes
LEFT JOIN _exponent._bronze_epic_clarity.zc_note_type_ip z
  ON z.TYPE_IP_C = h.IP_NOTE_TYPE_C
  AND z.DELETE_FLAG = 0

-- Resolve person_id from Epic person key
JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(
       CHR(31),
       'epic_clarity',
       'PATIENT',
       'PAT_ID',
       h.PAT_ID
     )
  AND stp.active_flag = TRUE

-- LEFT JOIN note type concept mapping using IP_NOTE_TYPE_C as source_id
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept nt_map
  ON nt_map.source_system = 'epic_clarity'
  AND nt_map.domain_id = 'Note Type'
  AND nt_map.source_id = CAST(h.IP_NOTE_TYPE_C AS STRING)
  AND nt_map.active_flag = TRUE

WHERE (h.DELETED_CAT_C IS NULL OR h.DELETED_CAT_C <> 2)
  AND h.DELETE_FLAG = 0
  AND h.PAT_ID IS NOT NULL
  AND h.ENTRY_DATETIME IS NOT NULL
  AND h.IP_NOTE_TYPE_C IS NOT NULL;

In [0]:
%sql
MERGE INTO _exponent.omop_silver.note AS target
USING (
  SELECT *
  FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY note_source_value
        ORDER BY note_date DESC
      ) AS rn
    FROM silver_note
  )
  WHERE rn = 1
) AS source
ON target.note_source_value = source.note_source_value

WHEN MATCHED AND NOT (
     target.person_id                     <=> source.person_id
 AND target.note_date                     <=> source.note_date
 AND target.note_datetime                 <=> source.note_datetime
 AND target.note_type_concept_id          <=> source.note_type_concept_id
 AND target.note_class_concept_id         <=> source.note_class_concept_id
 AND target.note_title                    <=> source.note_title
 AND target.note_text                     <=> source.note_text
 AND target.encoding_concept_id           <=> source.encoding_concept_id
 AND target.language_concept_id           <=> source.language_concept_id
 AND target.provider_id                   <=> source.provider_id
 AND target.visit_occurrence_source_value <=> source.visit_occurrence_source_value
 AND target.visit_detail_id               <=> source.visit_detail_id
 AND target.note_event_id                <=> source.note_event_id
 AND target.note_event_field_concept_id  <=> source.note_event_field_concept_id
 AND target.source_system                 <=> source.source_system
) THEN UPDATE SET
  target.person_id                     = source.person_id,
  target.note_date                     = source.note_date,
  target.note_datetime                 = source.note_datetime,
  target.note_type_concept_id          = source.note_type_concept_id,
  target.note_class_concept_id         = source.note_class_concept_id,
  target.note_title                    = source.note_title,
  target.note_text                     = source.note_text,
  target.encoding_concept_id           = source.encoding_concept_id,
  target.language_concept_id           = source.language_concept_id,
  target.provider_id                   = source.provider_id,
  target.visit_occurrence_source_value = source.visit_occurrence_source_value,
  target.visit_detail_id               = source.visit_detail_id,
  target.note_event_id                 = source.note_event_id,
  target.note_event_field_concept_id   = source.note_event_field_concept_id,
  target.source_system                 = source.source_system,
  target.last_mod_tsp                  = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  note_source_value,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_source_value,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  source_system,
  last_mod_tsp
) VALUES (
  source.note_source_value,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_source_value,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_note (
    source_system,
    note_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.note_source_value,
    TRUE                AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    CURRENT_TIMESTAMP() AS last_mod_tsp,
    NULL                AS merge_id,
    NULL                AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        note_source_value
    FROM _exponent.omop_silver.note
    WHERE note_source_value IS NOT NULL
      AND source_system = 'epic_clarity'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_note x
  ON s.note_source_value = x.note_source_value
 AND x.source_system = 'epic_clarity';

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold AS
SELECT
  stn.note_id,
  n.person_id,
  n.note_date,
  n.note_datetime,
  n.note_type_concept_id,
  n.note_class_concept_id,
  n.note_title,
  n.note_text,
  n.encoding_concept_id,
  n.language_concept_id,
  n.provider_id,
  stvo.visit_occurrence_id,
  n.visit_detail_id,
  n.note_event_id,
  n.note_event_field_concept_id,
  n.note_source_value

FROM _exponent.omop_silver.note n

JOIN _exponent.omop_mapping.source_to_note stn
  ON n.note_source_value = stn.note_source_value
 AND stn.source_system   = 'epic_clarity'
 AND stn.active_flag     = TRUE

-- Ensure person_id exists in gold person table (prevents orphan records)
INNER JOIN _exponent.omop_epic.person p
  ON p.person_id = n.person_id

-- Left join to death for after-death filter
LEFT JOIN _exponent.omop_epic.death d
  ON d.person_id = n.person_id

LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
  ON n.visit_occurrence_source_value = stvo.visit_occurrence_source_value
 AND stvo.source_system = 'epic_clarity'
 AND stvo.active_flag   = TRUE

WHERE n.source_system = 'epic_clarity'
  -- Exclude notes after death (plausibility check)
  AND (d.death_date IS NULL OR n.note_date <= d.death_date);

In [ ]:
%sql
MERGE INTO _exponent.omop_epic.note AS target
USING (
  SELECT
    g.*,
    ROW_NUMBER() OVER (
      PARTITION BY g.note_id
      ORDER BY vo.visit_occurrence_id DESC NULLS LAST
    ) AS rn
  FROM (
    SELECT
      stn.note_id,
      n.person_id,
      n.note_date,
      n.note_datetime,
      n.note_type_concept_id,
      n.note_class_concept_id,
      n.note_title,
      n.note_text,
      n.encoding_concept_id,
      n.language_concept_id,
      n.provider_id,
      stvo.visit_occurrence_id,
      n.visit_detail_id,
      n.note_event_id,
      n.note_event_field_concept_id,
      n.note_source_value

    FROM _exponent.omop_silver.note n

    JOIN _exponent.omop_mapping.source_to_note stn
      ON n.note_source_value = stn.note_source_value
      AND stn.source_system   = 'epic_clarity'
      AND stn.active_flag     = TRUE

    INNER JOIN _exponent.omop_epic.person p
      ON p.person_id = n.person_id

    LEFT JOIN _exponent.omop_epic.death d
      ON d.person_id = n.person_id

    LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
      ON n.visit_occurrence_source_value = stvo.visit_occurrence_source_value
      AND stvo.source_system = 'epic_clarity'
      AND stvo.active_flag   = TRUE

    WHERE n.source_system = 'epic_clarity'
      AND (d.death_date IS NULL OR n.note_date <= d.death_date)
  ) g
  LEFT JOIN _exponent.omop_epic.visit_occurrence vo
    ON vo.visit_occurrence_id = g.visit_occurrence_id
) AS source
ON target.note_id = source.note_id
WHERE source.rn = 1

WHEN MATCHED AND NOT (
     target.person_id             <=> source.person_id
 AND target.note_date             <=> source.note_date
 AND target.note_datetime         <=> source.note_datetime
 AND target.note_type_concept_id  <=> source.note_type_concept_id
 AND target.note_class_concept_id <=> source.note_class_concept_id
 AND target.note_title            <=> source.note_title
 AND target.note_text             <=> source.note_text
 AND target.encoding_concept_id   <=> source.encoding_concept_id
 AND target.language_concept_id   <=> source.language_concept_id
 AND target.provider_id           <=> source.provider_id
 AND target.visit_occurrence_id   <=> source.visit_occurrence_id
 AND target.visit_detail_id       <=> source.visit_detail_id
 AND target.note_event_id         <=> source.note_event_id
 AND target.note_event_field_concept_id <=> source.note_event_field_concept_id
 AND target.note_source_value     <=> source.note_source_value
) THEN UPDATE SET
  target.person_id             = source.person_id,
  target.note_date             = source.note_date,
  target.note_datetime         = source.note_datetime,
  target.note_type_concept_id  = source.note_type_concept_id,
  target.note_class_concept_id = source.note_class_concept_id,
  target.note_title            = source.note_title,
  target.note_text             = source.note_text,
  target.encoding_concept_id   = source.encoding_concept_id,
  target.language_concept_id   = source.language_concept_id,
  target.provider_id           = source.provider_id,
  target.visit_occurrence_id   = source.visit_occurrence_id,
  target.visit_detail_id       = source.visit_detail_id,
  target.note_event_id         = source.note_event_id,
  target.note_event_field_concept_id = source.note_event_field_concept_id,
  target.note_source_value     = source.note_source_value

WHEN NOT MATCHED THEN INSERT (
  note_id,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  note_source_value
) VALUES (
  source.note_id,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.note_source_value
);

In [ ]:
%sql
MERGE INTO _exponent.omop_epic.note AS target
USING (
  SELECT *
  FROM (
    SELECT
      g.*,
      ROW_NUMBER() OVER (
        PARTITION BY g.note_id
        ORDER BY vo.visit_occurrence_id DESC NULLS LAST
      ) AS rn
    FROM (
      SELECT
        stn.note_id,
        n.person_id,
        n.note_date,
        n.note_datetime,
        n.note_type_concept_id,
        n.note_class_concept_id,
        n.note_title,
        n.note_text,
        n.encoding_concept_id,
        n.language_concept_id,
        n.provider_id,
        stvo.visit_occurrence_id,
        n.visit_detail_id,
        n.note_event_id,
        n.note_event_field_concept_id,
        n.note_source_value

      FROM _exponent.omop_silver.note n

      JOIN _exponent.omop_mapping.source_to_note stn
        ON n.note_source_value = stn.note_source_value
        AND stn.source_system   = 'epic_clarity'
        AND stn.active_flag     = TRUE

      INNER JOIN _exponent.omop_epic.person p
        ON p.person_id = n.person_id

      LEFT JOIN _exponent.omop_epic.death d
        ON d.person_id = n.person_id

      LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
        ON n.visit_occurrence_source_value = stvo.visit_occurrence_source_value
        AND stvo.source_system = 'epic_clarity'
        AND stvo.active_flag   = TRUE

      WHERE n.source_system = 'epic_clarity'
        AND (d.death_date IS NULL OR n.note_date <= d.death_date)
    ) g
    LEFT JOIN _exponent.omop_epic.visit_occurrence vo
      ON vo.visit_occurrence_id = g.visit_occurrence_id
  )
  WHERE rn = 1
) AS source
ON target.note_id = source.note_id

WHEN MATCHED AND NOT (
     target.person_id             <=> source.person_id
 AND target.note_date             <=> source.note_date
 AND target.note_datetime         <=> source.note_datetime
 AND target.note_type_concept_id  <=> source.note_type_concept_id
 AND target.note_class_concept_id <=> source.note_class_concept_id
 AND target.note_title            <=> source.note_title
 AND target.note_text             <=> source.note_text
 AND target.encoding_concept_id   <=> source.encoding_concept_id
 AND target.language_concept_id   <=> source.language_concept_id
 AND target.provider_id           <=> source.provider_id
 AND target.visit_occurrence_id   <=> source.visit_occurrence_id
 AND target.visit_detail_id       <=> source.visit_detail_id
 AND target.note_event_id         <=> source.note_event_id
 AND target.note_event_field_concept_id <=> source.note_event_field_concept_id
 AND target.note_source_value     <=> source.note_source_value
) THEN UPDATE SET
  target.person_id             = source.person_id,
  target.note_date             = source.note_date,
  target.note_datetime         = source.note_datetime,
  target.note_type_concept_id  = source.note_type_concept_id,
  target.note_class_concept_id = source.note_class_concept_id,
  target.note_title            = source.note_title,
  target.note_text             = source.note_text,
  target.encoding_concept_id   = source.encoding_concept_id,
  target.language_concept_id   = source.language_concept_id,
  target.provider_id           = source.provider_id,
  target.visit_occurrence_id   = source.visit_occurrence_id,
  target.visit_detail_id       = source.visit_detail_id,
  target.note_event_id         = source.note_event_id,
  target.note_event_field_concept_id = source.note_event_field_concept_id,
  target.note_source_value     = source.note_source_value

WHEN NOT MATCHED THEN INSERT (
  note_id,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  note_source_value
) VALUES (
  source.note_id,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.note_source_value
);